# Unified Hidden-State Probe v4.1 — Master Notebook

**Research question:** given a frozen hidden-state representation, how recoverable is the clean dataset target from each layer?

The notebook deliberately does **not** preprocess, clean, or reload the datasets itself. `Get_Go_Emo.py` / `Get_Isear.py` remain the dataset-processing layer. This notebook imports their already-clean dataframes and passes those dataframes directly into the probe engine.

The source model is never retrained during probing. Only a separate supervised decoder is trained on a selected hidden-state layer.


## Data flow

```text
Get_Go_Emo.get_go() / Get_Isear.get_isr()
                |
                v
        clean dataframe
        clean_text / labels
                |
                v
       ExtractionArtifact
       hidden_states.npy
                |
                v
      row-count + provenance checks
                |
                v
        frozen train/val/test split
                |
                v
      layer -> probe -> target
                |
                v
  metrics + shuffled-label control
                |
                v
 model x dataset x layer x probe x repeat
```

The probe engine expects hidden states shaped `[samples, layers, hidden]`. The cleaned dataframe is the target source and is passed explicitly rather than discovered by the probing module.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from unified_hidden_state_probe_v4_1 import (
    ExtractionArtifact,
    DatasetContract,
    ProbeSpec,
    SplitConfig,
    AnalysisConfig,
    UnifiedProbeAnalyzer,
    GOEMOTIONS_CLASSES,
    ISEAR_CLASSES,
    run_matrix,
)

# -----------------------------------------------------------------------------
# Runtime controls
# -----------------------------------------------------------------------------
# DEBUG_MODE=True enables deep diagnostic output.
# DEBUG_MODE=False keeps the notebook runtime clean.
# Levels:
#   0 = clean / no routine logging
#   1 = lifecycle + final summary
#   2 = repeat + layer progress
#   3 = every probe fit + test metrics
DEBUG_MODE = False
VERBOSE = 3 if DEBUG_MODE else 0

EXTERNAL_ROOT = Path('/Volumes/Amirali/hidden_states')
EXPERIMENT_ID = 'baseline_v5_001'
MODEL_NAME = 'google-bert/bert-base-uncased'
DATASET_NAME = 'goEmo'

DATASET_DIR = (
    EXTERNAL_ROOT / 'experiments' / EXPERIMENT_ID / 'models'
    / Path(*MODEL_NAME.split('/')) / 'datasets' / DATASET_NAME
)

print('Probe module configured.') if DEBUG_MODE else None


## Load the clean dataset directly

This is intentionally the lightweight interface. The probe system does not need to know where the CSVs came from or how emojis/emoticons were cleaned.

For GoEmotions the expected dataframe contract is:

- `clean_text`: the exact text sequence used by extraction
- `labels`: the target sequence

For ISEAR the analogous contract can use `text` and `emotion`.


In [ ]:
from Get_Go_Emo import get_go
from Get_Isear import get_isr

go_df = get_go()
isear_df = get_isr()

required_go_columns = {'clean_text', 'labels'}
missing = required_go_columns - set(go_df.columns)
if missing:
    raise RuntimeError(f'GoEmotions clean dataset is missing required columns: {sorted(missing)}')

print('GoEmotions rows:', len(go_df)) if DEBUG_MODE else None
print('ISEAR rows:', len(isear_df)) if DEBUG_MODE else None


### Input preview — clean datasets
This is the exact data handed to the probe layer. The preview is intentionally small; it is a sanity check, not a second copy of the dataset.


In [ ]:
print("GoEmotions input →")
display(go_df.head(5))
print("ISEAR input →")
display(isear_df.head(5))


## Dataset contracts

The contract now contains only target interpretation metadata. It does not tell the probe module to preprocess the dataset again because the clean dataframe has already been produced.


In [ ]:
goemotions_contract = DatasetContract(
    target_type='goemotions',
    text_column='clean_text',
    label_column='labels',
    task_type='multi_label',
    label_format='auto',
    class_order=GOEMOTIONS_CLASSES,
    require_provenance=False,
    require_label_fingerprint=False,
)

isear_contract = DatasetContract(
    target_type='isear',
    text_column='text',
    label_column='emotion',
    task_type='single_label',
    label_format='scalar',
    class_order=ISEAR_CLASSES,
    require_provenance=False,
    require_label_fingerprint=False,
)

print('Contracts ready.') if DEBUG_MODE else None


## Probe family

The linear logistic probe is the primary low-complexity diagnostic. It asks how much target information is linearly recoverable. The MLPs test controlled nonlinear decodability.

Keep these probe definitions fixed across layers and models when making cross-layer claims.


In [ ]:
probes = [
    ProbeSpec(
        name='linear_logistic', type='logistic', complexity='linear',
        standardize=True, C=1.0, max_iter=3000,
        selection_metric='macro_f1',
    ),
    ProbeSpec(
        name='mlp_1_hidden', type='mlp', complexity='1_hidden',
        standardize=True, hidden_dims=['0.5d'], learning_rate=1e-3,
        weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
        selection_metric='macro_f1',
    ),
    ProbeSpec(
        name='mlp_2_hidden', type='mlp', complexity='2_hidden',
        standardize=True, hidden_dims=['0.5d', '0.25d'], learning_rate=1e-3,
        weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
        selection_metric='macro_f1',
    ),
    ProbeSpec(
        name='mlp_3_hidden', type='mlp', complexity='3_hidden',
        standardize=True, hidden_dims=['0.5d', '0.25d', '0.125d'], learning_rate=1e-3,
        weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
        selection_metric='macro_f1',
    ),
]


### Input preview — probe configuration
Each row is one decoder family and its explicit complexity.


In [ ]:
probe_preview = pd.DataFrame([
    {"name": p.name, "type": p.type, "complexity": p.complexity, "hidden_dims": p.hidden_dims, "epochs": p.epochs}
    for p in probes
])
display(probe_preview)


## Pilot analysis configuration

Start with a pilot while checking the alignment and result curves. Once the pilot is validated, change `max_samples=None` for the complete extracted dataset.


In [ ]:
config = AnalysisConfig(
    dataset=goemotions_contract,
    probes=probes,
    layers='all',
    split=SplitConfig(train=0.80, validation=0.10, test=0.10, seed=42, stratify=True),
    repeats=3,
    max_samples=5000,
    shuffled_label_control=True,
    shuffled_control_repeats=3,
    run_control_on_all_layers=True,
    pca_enabled=True,
    silhouette_enabled=True,
    pca_samples=3000,
    silhouette_samples=3000,
    enable_per_class_metrics=True,
    enable_feature_statistics=True,
    verbose=VERBOSE,
    score_weights={
        'macro_f1': 0.25,
        'balanced_accuracy': 0.15,
        'mcc': 0.15,
        'log_loss_score': 0.10,
        'selectivity': 0.20,
        'stability': 0.10,
        'geometry': 0.05,
    },
)


### Input preview — experiment configuration


In [ ]:
config_preview = pd.DataFrame([{
    "layers": config.layers, "repeats": config.repeats, "max_samples": config.max_samples,
    "train": config.split.train, "validation": config.split.validation, "test": config.split.test,
    "shuffled_controls": config.shuffled_control_repeats, "verbose": config.verbose
}])
display(config_preview)


## Artifact validation and explicit dataset-to-state alignment

The artifact is loaded from external storage. The probe engine requires:

- `[N, L, D]` hidden states,
- one completion flag per sample,
- metadata `N` agreeing with the stored array,
- finite representations,
- a clean dataframe with exactly `N` rows.

Old artifacts with no provenance hash are reported as **unverified**, not incorrectly called a mismatch. A real stored hash disagreement still stops the run.


In [ ]:
artifact = ExtractionArtifact(DATASET_DIR)

if len(go_df) != artifact.sample_count:
    raise RuntimeError(
        f'Clean GoEmotions rows ({len(go_df)}) != hidden-state samples ({artifact.sample_count}).'
    )

OUTPUT_DIR = DATASET_DIR / 'analysis' / 'probes' / 'notebook_v4_master'

analyzer = UnifiedProbeAnalyzer(
    artifact=artifact,
    config=config,
    output_dir=OUTPUT_DIR,
    dataset_df=go_df,
)

if DEBUG_MODE:
    print(json.dumps(artifact.analysis_summary(), indent=2, default=str))
    print(json.dumps(analyzer.text_alignment, indent=2, default=str))
    print(json.dumps(analyzer.label_alignment, indent=2, default=str))


### Input preview — hidden-state artifact
This confirms the dimensions and the first few dataset rows before training starts.


In [ ]:
artifact_preview = pd.DataFrame([
    {"model": artifact.model_name, "dataset": artifact.dataset_name, "N": artifact.sample_count, "L": artifact.hidden_layers, "D": artifact.hidden_size, "pooling": artifact.pooling}
])
display(artifact_preview)
display(go_df[["clean_text", "labels"]].head(5))


## Where probing happens

The actual supervised decoding happens inside `UnifiedProbeAnalyzer.run()`.

For each layer it loads only the current repeat's selected rows from the memory-mapped hidden-state artifact, creates fixed train/validation/test membership, standardizes using training data only, and calls `fit_probe(...)`.

The original source model is never retrained or modified.


In [ ]:
# ACTUAL PROBE RUN
# Input: validated hidden states + clean targets + fixed experimental configuration.
# Output: one row per repeat × layer × probe, plus best-layer aggregation.
results_df, best_df = analyzer.run()

## Inspect the complete layer-wise result table


In [ ]:

print(f"Probe output rows: {len(results_df)}")
print("Per-run output preview →")
display(results_df.head(10))
print("Best-layer output preview →")
display(best_df.head(10))

In [ ]:
result_columns = [
    'probe','probe_complexity','layer_index','relative_layer_depth','repeat',
    'input_dim','parameters','test_macro_f1','test_balanced_accuracy',
    'test_mcc','test_log_loss','selectivity','complexity_penalty','probe_score',
]

view = (
    results_df[result_columns]
    .sort_values(['probe', 'layer_index', 'repeat'])
)
display(view.head(30))

# The multi-label metrics are now accompanied by coverage diagnostics so an
# undefined ROC-AUC/AP value can be understood rather than hidden behind a warning.
coverage_cols = [
    c for c in [
        'probe','layer_index','repeat','test_macro_f1',
        'test_roc_auc_macro','test_average_precision_macro',
        'test_labels_with_positive_support','test_labels_with_both_support'
    ] if c in results_df.columns
]
if coverage_cols:
    print('Metric-coverage diagnostics →')
    display(results_df[coverage_cols].head(20))


## Alignment artifact: what should be stored for future extractions?

The safest long-term design is **not** to hide labels inside the floating-point tensor. Instead keep a small aligned sidecar next to it:

```text
hidden_states.npy          [N, L, D]
completed.npy              [N]
row_ids.npy                [N]
labels.npy                 [N] or [N, C]
alignment.json
extraction.json
```

`row_ids.npy` is especially important. If a 34-row batch accidentally produces 32 states, the extractor must stop before committing that batch. If a later bug reorders or skips rows, the stored source row IDs expose the mismatch.

The probe itself can save `probe_alignment_manifest.json`, but that is a probe-time record. It cannot retroactively prove what the extractor consumed. Future extraction runs should create the alignment manifest **during extraction**.


## Generalising to the 25 frozen models

The cleanest matrix entry is now:

```python
{
    'model': '...',
    'dataset': 'goEmo',
    'contract': goemotions_contract,
    'dataset_df': go_df,
}
```

The same `go_df` object can be supplied for every model, avoiding repeated dataset loading and preprocessing.


In [ ]:
RUN_MATRIX = [
    {
        'model': 'google-bert/bert-base-uncased',
        'dataset': 'goEmo',
        'contract': goemotions_contract,
        'dataset_df': go_df,
    },
    {
        'model': 'Qwen/Qwen2-0.5B',
        'dataset': 'goEmo',
        'contract': goemotions_contract,
        'dataset_df': go_df,
    },
]

# Use VERBOSE=0 for a clean 25-model sweep, VERBOSE=1 for progress.
matrix_results = run_matrix(
    RUN_MATRIX,
    external_root=EXTERNAL_ROOT,
    experiment_id=EXPERIMENT_ID,
    probes=probes,
    repeats=3,
    max_samples=5000,
    verbose=VERBOSE,
)

print('Matrix output preview →')
display(matrix_results.head(10))


## Interpretability roadmap — second stage

**LIME:** wrap `text -> frozen model -> chosen layer -> probe -> probability`, perturb the text, and inspect which words most change the probe prediction. This explains what textual evidence the probe exploits.

**SHAP:** similar objective with Shapley-style additive attributions; useful for deeper token-level analysis once the layer-wise benchmark is stable.

**Integrated Gradients:** differentiable attribution through a PyTorch model + layer + probe path. Good when token-level gradient explanations are needed.

**TCAV:** create concept sets (e.g. joy/fear examples), learn concept directions in activation space, and ask where those concepts become strongly represented.

**CKA:** compare representation geometry across layers and across the 25 models, especially when hidden widths and depths differ.

**Activation patching/interventions:** later-stage causal work. Probe results establish decodability first; intervention asks whether manipulating a promising representation changes model behavior.


## Scientific interpretation guardrails

A reproducible rise and fall in probe performance across layers is meaningful evidence about **recoverability** of the target from the representation.

It does not by itself prove:

- that a layer exclusively learns emotion,
- that earlier layers are exclusively linguistic,
- that the source model uses the decoded emotion information causally.

To support stronger specialization claims, repeat the same probing framework with additional targets such as lexical/syntactic/semantic controls and compare their layer profiles.


## Metric warnings: what you should expect now
For multi-label emotion data, a particular test split can contain zero positives for a rare emotion. ROC-AUC is undefined unless both positive and negative examples exist. Average precision is also undefined when there is no positive class. v4.1 explicitly skips undefined class-wise ROC-AUC/AP terms and records coverage (`valid_auc_labels`, `valid_ap_labels`) instead of flooding the notebook with sklearn warnings. Macro-F1, Hamming score, MCC and the other defined metrics remain available.
